In [ ]:
# Gera arquivos .parquet + parquet + snappy (compressao)

from pathlib import Path
from zipfile import ZipFile
from lxml import etree
import os
import glob
from datetime import datetime
import pandas as pd  
import pyarrow as pa
import pyarrow.parquet as pq

# Caminho para o dump
base_dir = Path(r"C:\Dumps\todos_enms")

# Buscar todos os arquivos ZIP na pasta
zip_files = list(base_dir.glob("*.zip"))

# Descompactar todos os ZIPs
for zip_path in zip_files:
    with ZipFile(zip_path, "r") as zf:
        zf.extractall(base_dir)
    print(f"Extraído: {zip_path} -> {base_dir}")

def parse_single_ericsson_xml(xml_path, output_dir):
    """
    Processa um único arquivo XML e gera arquivos Parquet (streaming) com timestamp.
    """
    ns = {
        'xn': 'genericNrm.xsd',
        'es': 'EricssonSpecificAttributes.xsd'
    }
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Gerar timestamp para este XML (DD_HH_MM_SS)
    timestamp = datetime.now().strftime("%d_%H_%M_%S")
    
    print(f"\n========== PROCESSANDO: {xml_path} ==========")
    print(f"Timestamp: {timestamp}")
    
    # ========== ESTRUTURAS ==========
    # Acúmulo por MO em lotes + controle de schema/escrita parquet
    BATCH_SIZE = 100_000  # ajuste se quiser lotes maiores/menores

    buffers = {}           # mo_name -> list[dict] (lote em memória)
    all_columns = {}       # mo_name -> set(str) (todas as colunas já vistas)
    data_columns = {}      # mo_name -> set(str) (mantido como no seu código)
    writers = {}           # mo_name -> pq.ParquetWriter
    schema_cols = {}       # mo_name -> list[str] (ordem/definição atual do schema do writer)
    part_idx = {}          # mo_name -> int (contador de partes)
    # ================================

    def _ensure_writer(mo_name, xml_filename_stem):
        """Garante que existe um writer para o MO com o schema atual (all_columns[mo]). 
           Se já existir mas o schema mudou, fecha e reabre em um novo part."""
        cols_sorted = sorted(all_columns[mo_name])

        # Se não há writer ainda -> cria
        if mo_name not in writers:
            part_idx[mo_name] = 0
            out_path = Path(output_dir) / f"{mo_name}_{xml_filename_stem}_{timestamp}_part{part_idx[mo_name]:05d}.parquet"
            # Schema: tudo string (rápido de escrever; você tipa na leitura)
            fields = [pa.field(c, pa.string()) for c in cols_sorted]
            schema = pa.schema(fields)
            writers[mo_name] = pq.ParquetWriter(
                out_path, schema,
                compression="snappy",
                use_dictionary=True,
                write_statistics=True
            )
            schema_cols[mo_name] = cols_sorted
            print(f"🧷 Writer aberto: {out_path} (cols={len(cols_sorted)})")
            return

        # Existe writer, mas schema mudou? Fecha e abre novo part
        if schema_cols[mo_name] != cols_sorted:
            # fecha atual
            try:
                writers[mo_name].close()
            except Exception:
                pass
            part_idx[mo_name] += 1
            out_path = Path(output_dir) / f"{mo_name}_{xml_filename_stem}_{timestamp}_part{part_idx[mo_name]:05d}.parquet"
            fields = [pa.field(c, pa.string()) for c in cols_sorted]
            schema = pa.schema(fields)
            writers[mo_name] = pq.ParquetWriter(
                out_path, schema,
                compression="snappy",
                use_dictionary=True,
                write_statistics=True
            )
            schema_cols[mo_name] = cols_sorted
            print(f"🔁 Schema evoluiu; novo writer: {out_path} (cols={len(cols_sorted)})")

    def _flush(mo_name, xml_filename_stem):
        """Escreve o lote atual do MO como um row group e limpa buffer."""
        buf = buffers.get(mo_name, [])
        if not buf:
            return
        # Garante writer com schema atualizado
        _ensure_writer(mo_name, xml_filename_stem)
        cols = schema_cols[mo_name]
        # Constrói Table coluna-a-coluna (string)
        arrays = []
        for c in cols:
            col_data = []
            for r in buf:
                v = r.get(c)
                if v is None:
                    col_data.append(None)
                else:
                    col_data.append(str(v))
            arrays.append(pa.array(col_data, type=pa.string()))
        table = pa.Table.from_arrays(arrays, names=cols)
        writers[mo_name].write_table(table)  # um row group por flush
        buffers[mo_name].clear()

    def _close_all_writers():
        # 1) Flush final de TODOS os MOs com dados no buffer (mesmo sem writer aberto ainda)
        for mo in list(buffers.keys()):
            if buffers[mo]:
                _flush(mo, xml_filename)  # _flush abre o writer se necessário

        # 2) Fecha todos os writers existentes
        for mo, w in list(writers.items()):
            try:
                w.close()
            except Exception:
                pass

        # (opcional) limpar estruturas
        buffers.clear()
        writers.clear()

    # ========== PROCESSA O XML ATUAL ==========
    context = etree.iterparse(str(xml_path), events=("end",), tag="{genericNrm.xsd}VsDataContainer")
    xml_filename = xml_path.stem  # Nome do arquivo XML sem extensão
    
    mo_types_with_data = set()

    try:
        for event, elem in context:
            # Contextos (site/eNB)
            me_context_elem = elem.xpath("ancestor::xn:MeContext", namespaces=ns)
            managed_element_elem = elem.xpath("ancestor::xn:ManagedElement", namespaces=ns)
            
            me_context = me_context_elem[0].get("id") if me_context_elem else ""
            managed_element = managed_element_elem[0].get("id") if managed_element_elem else ""
            if managed_element == "1":
                managed_element = me_context
            
            mo_type_elem = elem.find("xn:attributes/xn:vsDataType", ns)
            if mo_type_elem is not None:
                mo_name = mo_type_elem.text.strip().replace("vsData", "")
                
                # Inicializa estruturas
                if mo_name not in buffers:
                    buffers[mo_name] = []
                    all_columns[mo_name] = set(["eNB", "ManagedElement", "MO"])
                    data_columns[mo_name] = set()
                
                parameters = {
                    "eNB": me_context,
                    "ManagedElement": managed_element,
                    "MO": mo_name
                }

                # DETECÇÃO ESTRUTURAL
                ancestors = elem.xpath("ancestor::xn:VsDataContainer", namespaces=ns)
                for ancestor in ancestors:
                    ancestor_id = ancestor.get("id") or ""
                    if ancestor_id:
                        vs_data_type_elem = ancestor.find("xn:attributes/xn:vsDataType", ns)
                        if vs_data_type_elem is not None:
                            vs_data_type = vs_data_type_elem.text.strip()
                            if vs_data_type in ("vsDataEUtranCellFDD", "vsDataEUtranCellTDD"):
                                parameters["EUtranCell"] = ancestor_id
                                all_columns[mo_name].add("EUtranCell")
                            elif vs_data_type in ("vsDataNRCellCU", "vsDataNRCellDU"):
                                parameters["NRCell"] = ancestor_id
                                all_columns[mo_name].add("NRCell")
                            elif vs_data_type == "vsDataENodeBFunction":
                                parameters["ENodeBFunction"] = ancestor_id
                                all_columns[mo_name].add("ENodeBFunction")
                            elif vs_data_type == "vsDataGNBCUCPFunction":
                                parameters["GNBCUCPFunction"] = ancestor_id
                                all_columns[mo_name].add("GNBCUCPFunction")
                            elif vs_data_type == "vsDataGNBDUFunction":
                                parameters["GNBDUFunction"] = ancestor_id
                                all_columns[mo_name].add("GNBDUFunction")
                            elif vs_data_type == "vsDataEUtranFreqRelation":
                                parameters["EUtranFreqRelation"] = ancestor_id
                                all_columns[mo_name].add("EUtranFreqRelation")
                            elif vs_data_type == "vsDataNRFreqRelation":
                                parameters["NRFreqRelation"] = ancestor_id
                                all_columns[mo_name].add("NRFreqRelation")
                            elif vs_data_type.startswith("vsData") and vs_data_type != f"vsData{mo_name}":
                                column_name = vs_data_type.replace("vsData", "")
                                parameters[column_name] = ancestor_id
                                all_columns[mo_name].add(column_name)

                # Coleta dos parâmetros do bloco vsData específico do MO atual
                vs_data_element = elem.find(f"xn:attributes/es:{mo_type_elem.text}", ns)
                record_has_data = False
                if vs_data_element is not None:
                    for param in vs_data_element:
                        param_name = param.tag.split('}')[-1]
                        
                        if len(param):  # tem filhos
                            for sub_param in param:
                                sub_param_name = sub_param.tag.split('}')[-1]
                                sub_param_value = (sub_param.text or "").strip()
                                combined_name = f"{param_name}_{sub_param_name}"
                                data_columns[mo_name].add(combined_name)
                                all_columns[mo_name].add(combined_name)
                                if sub_param_value != "":
                                    parameters[combined_name] = sub_param_value
                                    record_has_data = True
                        else:
                            param_value = (param.text or "").strip()
                            data_columns[mo_name].add(param_name)
                            all_columns[mo_name].add(param_name)
                            if param_value != "":
                                parameters[param_name] = param_value
                                record_has_data = True

                if record_has_data:
                    mo_types_with_data.add(mo_name)
                    buffers[mo_name].append(parameters)
                    # flush por lote
                    if len(buffers[mo_name]) >= BATCH_SIZE:
                        _flush(mo_name, xml_filename)
            
            # Libera memória do elemento já processado
            elem.clear()
    finally:
        # Libera parser
        try:
            del context
        except Exception:
            pass
        # Flush final + fechar writers (AGORA flusha todos, mesmo sem writer aberto)
        _close_all_writers()
    
    # Log final
    print(f"\n========== XML {xml_filename} PROCESSADO ==========")
    print(f"Total de tipos de MO com dados: {len(mo_types_with_data)}")
    for mo_name in sorted(mo_types_with_data):
        print(f"  - {mo_name}")
    
    return len(mo_types_with_data)

# ========== PROGRAMA PRINCIPAL ==========
def main():
    directory_path = base_dir
    output_path = base_dir
    
    # Buscar todos os arquivos XML na pasta
    xml_files = list(directory_path.glob("*.xml"))
    
    if not xml_files:
        print("⚠️  Nenhum arquivo XML encontrado na pasta!")
        return
    
    print(f"📁 Encontrados {len(xml_files)} arquivos XML para processar:")
    for xml_file in xml_files:
        print(f"   - {xml_file.name}")
    
    total_xml_processed = 0
    total_mo_types = 0
    
    # Processar cada XML individualmente
    for xml_file in xml_files:
        print(f"\n{'='*80}")
        print(f"🔄 INICIANDO PROCESSAMENTO DO XML {total_xml_processed + 1}/{len(xml_files)}")
        print(f"📄 Arquivo: {xml_file.name}")
        print(f"{'='*80}")
        
        try:
            mo_types_count = parse_single_ericsson_xml(xml_file, output_path)
            total_xml_processed += 1
            total_mo_types += mo_types_count
            
            print(f"✅ XML {xml_file.name} processado com sucesso!")
            
        except Exception as e:
            print(f"❌ Erro ao processar {xml_file.name}: {str(e)}")
            continue
    
    # Resumo final
    print(f"\n{'='*80}")
    print(f"🎉 PROCESSAMENTO CONCLUÍDO!")
    print(f"📊 Resumo:")
    print(f"   - Total de XMLs processados: {total_xml_processed}/{len(xml_files)}")
    print(f"   - Total de tipos de MO encontrados: {total_mo_types}")
    print(f"   - Pasta de output: {output_path}")
    print(f"{'='*80}")

# Executar o programa
if __name__ == "__main__":
    main()


# Mesmo script, porem, expurgando vsData dos dados - rodou SP em 55min (3340 segundos)

# Gera arquivos .parquet + compressao

from pathlib import Path
from zipfile import ZipFile
from lxml import etree
import os
import glob
from datetime import datetime
import pandas as pd  # mantido (não usado diretamente, mas deixei como estava)
import pyarrow as pa
import pyarrow.parquet as pq
import re  # <-- adicionado

# Caminho para o dump
base_dir = Path(r"C:\MoB_IMP\Dumps")

# Buscar todos os arquivos ZIP na pasta
zip_files = list(base_dir.glob("*.zip"))

# Descompactar todos os ZIPs
for zip_path in zip_files:
    with ZipFile(zip_path, "r") as zf:
        zf.extractall(base_dir)
    print(f"Extraído: {zip_path} -> {base_dir}")

def parse_single_ericsson_xml(xml_path, output_dir):
    """
    Processa um único arquivo XML e gera arquivos Parquet (streaming) com timestamp.
    """
    ns = {
        'xn': 'genericNrm.xsd',
        'es': 'EricssonSpecificAttributes.xsd'
    }
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Gerar timestamp para este XML (DD_HH_MM_SS)
    timestamp = datetime.now().strftime("%d_%H_%M_%S")
    
    print(f"\n========== PROCESSANDO: {xml_path} ==========")
    print(f"Timestamp: {timestamp}")
    
    # ===== limpeza do prefixo "vsData" em valores =====
    VS_PAT = re.compile(r'\bvsData([A-Za-z0-9_]+)=')
    def _strip_vsdata_prefix(value: str) -> str:
        # Remove "vsData" de tokens do tipo vsDataAlgo=...
        if isinstance(value, str) and "vsData" in value and "=" in value:
            return VS_PAT.sub(r"\1=", value)
        return value
    # ==================================================
    
    # ========== ESTRUTURAS ==========
    # Acúmulo por MO em lotes + controle de schema/escrita parquet
    BATCH_SIZE = 100_000  # ajuste se quiser lotes maiores/menores

    buffers = {}           # mo_name -> list[dict] (lote em memória)
    all_columns = {}       # mo_name -> set(str) (todas as colunas já vistas)
    data_columns = {}      # mo_name -> set(str) (mantido como no seu código)
    writers = {}           # mo_name -> pq.ParquetWriter
    schema_cols = {}       # mo_name -> list[str] (ordem/definição atual do schema do writer)
    part_idx = {}          # mo_name -> int (contador de partes)
    # ================================

    def _ensure_writer(mo_name, xml_filename_stem):
        """Garante que existe um writer para o MO com o schema atual (all_columns[mo]). 
           Se já existir mas o schema mudou, fecha e reabre em um novo part."""
        cols_sorted = sorted(all_columns[mo_name])

        # Se não há writer ainda -> cria
        if mo_name not in writers:
            part_idx[mo_name] = 0
            out_path = Path(output_dir) / f"{mo_name}_{xml_filename_stem}_{timestamp}_part{part_idx[mo_name]:05d}.parquet"
            # Schema: tudo string (rápido de escrever; você tipa na leitura)
            fields = [pa.field(c, pa.string()) for c in cols_sorted]
            schema = pa.schema(fields)
            writers[mo_name] = pq.ParquetWriter(
                out_path, schema,
                compression="snappy",
                use_dictionary=True,
                write_statistics=True
            )
            schema_cols[mo_name] = cols_sorted
            print(f"🧷 Writer aberto: {out_path} (cols={len(cols_sorted)})")
            return

        # Existe writer, mas schema mudou? Fecha e abre novo part
        if schema_cols[mo_name] != cols_sorted:
            try:
                writers[mo_name].close()
            except Exception:
                pass
            part_idx[mo_name] += 1
            out_path = Path(output_dir) / f"{mo_name}_{xml_filename_stem}_{timestamp}_part{part_idx[mo_name]:05d}.parquet"
            fields = [pa.field(c, pa.string()) for c in cols_sorted]
            schema = pa.schema(fields)
            writers[mo_name] = pq.ParquetWriter(
                out_path, schema,
                compression="snappy",
                use_dictionary=True,
                write_statistics=True
            )
            schema_cols[mo_name] = cols_sorted
            print(f"🔁 Schema evoluiu; novo writer: {out_path} (cols={len(cols_sorted)})")

    def _flush(mo_name, xml_filename_stem):
        """Escreve o lote atual do MO como um row group e limpa buffer."""
        buf = buffers.get(mo_name, [])
        if not buf:
            return
        # Garante writer com schema atualizado
        _ensure_writer(mo_name, xml_filename_stem)
        cols = schema_cols[mo_name]
        # Constrói Table coluna-a-coluna (string)
        arrays = []
        for c in cols:
            col_data = []
            for r in buf:
                v = r.get(c)
                if v is None:
                    col_data.append(None)
                else:
                    col_data.append(str(v))
            arrays.append(pa.array(col_data, type=pa.string()))
        table = pa.Table.from_arrays(arrays, names=cols)
        writers[mo_name].write_table(table)  # um row group por flush
        buffers[mo_name].clear()

    def _close_all_writers():
        # 1) Flush final de TODOS os MOs com dados no buffer (mesmo sem writer aberto ainda)
        for mo in list(buffers.keys()):
            if buffers[mo]:
                _flush(mo, xml_filename)  # _flush abre o writer se necessário
        # 2) Fecha todos os writers existentes
        for mo, w in list(writers.items()):
            try:
                w.close()
            except Exception:
                pass
        buffers.clear()
        writers.clear()

    # ========== PROCESSA O XML ATUAL ==========
    context = etree.iterparse(str(xml_path), events=("end",), tag="{genericNrm.xsd}VsDataContainer")
    xml_filename = xml_path.stem  # Nome do arquivo XML sem extensão
    
    mo_types_with_data = set()

    try:
        for event, elem in context:
            # Contextos (site/eNB)
            me_context_elem = elem.xpath("ancestor::xn:MeContext", namespaces=ns)
            managed_element_elem = elem.xpath("ancestor::xn:ManagedElement", namespaces=ns)
            
            me_context = me_context_elem[0].get("id") if me_context_elem else ""
            managed_element = managed_element_elem[0].get("id") if managed_element_elem else ""
            if managed_element == "1":
                managed_element = me_context
            
            mo_type_elem = elem.find("xn:attributes/xn:vsDataType", ns)
            if mo_type_elem is not None:
                mo_name = mo_type_elem.text.strip().replace("vsData", "")
                
                # Inicializa estruturas
                if mo_name not in buffers:
                    buffers[mo_name] = []
                    all_columns[mo_name] = set(["eNB", "ManagedElement", "MO"])
                    data_columns[mo_name] = set()
                
                parameters = {
                    "eNB": me_context,
                    "ManagedElement": managed_element,
                    "MO": mo_name
                }

                # DETECÇÃO ESTRUTURAL
                ancestors = elem.xpath("ancestor::xn:VsDataContainer", namespaces=ns)
                for ancestor in ancestors:
                    ancestor_id = ancestor.get("id") or ""
                    if ancestor_id:
                        vs_data_type_elem = ancestor.find("xn:attributes/xn:vsDataType", ns)
                        if vs_data_type_elem is not None:
                            vs_data_type = vs_data_type_elem.text.strip()
                            if vs_data_type in ("vsDataEUtranCellFDD", "vsDataEUtranCellTDD"):
                                parameters["EUtranCell"] = ancestor_id
                                all_columns[mo_name].add("EUtranCell")
                            elif vs_data_type in ("vsDataNRCellCU", "vsDataNRCellDU"):
                                parameters["NRCell"] = ancestor_id
                                all_columns[mo_name].add("NRCell")
                            elif vs_data_type == "vsDataENodeBFunction":
                                parameters["ENodeBFunction"] = ancestor_id
                                all_columns[mo_name].add("ENodeBFunction")
                            elif vs_data_type == "vsDataGNBCUCPFunction":
                                parameters["GNBCUCPFunction"] = ancestor_id
                                all_columns[mo_name].add("GNBCUCPFunction")
                            elif vs_data_type == "vsDataGNBDUFunction":
                                parameters["GNBDUFunction"] = ancestor_id
                                all_columns[mo_name].add("GNBDUFunction")
                            elif vs_data_type == "vsDataEUtranFreqRelation":
                                parameters["EUtranFreqRelation"] = ancestor_id
                                all_columns[mo_name].add("EUtranFreqRelation")
                            elif vs_data_type == "vsDataNRFreqRelation":
                                parameters["NRFreqRelation"] = ancestor_id
                                all_columns[mo_name].add("NRFreqRelation")
                            elif vs_data_type.startswith("vsData") and vs_data_type != f"vsData{mo_name}":
                                column_name = vs_data_type.replace("vsData", "")
                                parameters[column_name] = ancestor_id
                                all_columns[mo_name].add(column_name)

                # Coleta dos parâmetros do bloco vsData específico do MO atual
                vs_data_element = elem.find(f"xn:attributes/es:{mo_type_elem.text}", ns)
                record_has_data = False
                if vs_data_element is not None:
                    for param in vs_data_element:
                        param_name = param.tag.split('}')[-1]
                        
                        if len(param):  # tem filhos
                            for sub_param in param:
                                sub_param_name = sub_param.tag.split('}')[-1]
                                sub_param_value = _strip_vsdata_prefix((sub_param.text or "").strip())  # <-- limpo
                                combined_name = f"{param_name}_{sub_param_name}"
                                data_columns[mo_name].add(combined_name)
                                all_columns[mo_name].add(combined_name)
                                if sub_param_value != "":
                                    parameters[combined_name] = sub_param_value
                                    record_has_data = True
                        else:
                            param_value = _strip_vsdata_prefix((param.text or "").strip())  # <-- limpo
                            data_columns[mo_name].add(param_name)
                            all_columns[mo_name].add(param_name)
                            if param_value != "":
                                parameters[param_name] = param_value
                                record_has_data = True

                if record_has_data:
                    mo_types_with_data.add(mo_name)
                    buffers[mo_name].append(parameters)
                    # flush por lote
                    if len(buffers[mo_name]) >= BATCH_SIZE:
                        _flush(mo_name, xml_filename)
            
            # Libera memória do elemento já processado
            elem.clear()
    finally:
        # Libera parser
        try:
            del context
        except Exception:
            pass
        # Flush final + fechar writers (flusha todos, mesmo sem writer aberto)
        _close_all_writers()
    
    # Log final
    print(f"\n========== XML {xml_filename} PROCESSADO ==========")
    print(f"Total de tipos de MO com dados: {len(mo_types_with_data)}")
    for mo_name in sorted(mo_types_with_data):
        print(f"  - {mo_name}")
    
    return len(mo_types_with_data)

# ========== PROGRAMA PRINCIPAL ==========
def main():
    directory_path = base_dir
    output_path = base_dir
    
    # Buscar todos os arquivos XML na pasta
    xml_files = list(directory_path.glob("*.xml"))
    
    if not xml_files:
        print("⚠️  Nenhum arquivo XML encontrado na pasta!")
        return
    
    print(f"📁 Encontrados {len(xml_files)} arquivos XML para processar:")
    for xml_file in xml_files:
        print(f"   - {xml_file.name}")
    
    total_xml_processed = 0
    total_mo_types = 0
    
    # Processar cada XML individualmente
    for xml_file in xml_files:
        print(f"\n{'='*80}")
        print(f"🔄 INICIANDO PROCESSAMENTO DO XML {total_xml_processed + 1}/{len(xml_files)}")
        print(f"📄 Arquivo: {xml_file.name}")
        print(f"{'='*80}")
        
        try:
            mo_types_count = parse_single_ericsson_xml(xml_file, output_path)
            total_xml_processed += 1
            total_mo_types += mo_types_count
            
            print(f"✅ XML {xml_file.name} processado com sucesso!")
            
        except Exception as e:
            print(f"❌ Erro ao processar {xml_file.name}: {str(e)}")
            continue
    
    # Resumo final
    print(f"\n{'='*80}")
    print(f"🎉 PROCESSAMENTO CONCLUÍDO!")
    print(f"📊 Resumo:")
    print(f"   - Total de XMLs processados: {total_xml_processed}/{len(xml_files)}")
    print(f"   - Total de tipos de MO encontrados: {total_mo_types}")
    print(f"   - Pasta de output: {output_path}")
    print(f"{'='*80}")

# Executar o programa
if __name__ == "__main__":
    main()